# Build open-deposit files from the aggregated GeoPackage

Produces the files for the data repository (Zenodo/Figshare):

1. **`harvest_density_long.csv[.gz]`** — a tidy, `hex_id`-linked density table (annual + multi-year), **no geometry**.
2. **`hexgrid_geometry.gpkg`** — geometry-only hexagon layer (`hex_id` + area) in native CRS (EPSG:5514).
3. **`hexgrid_geometry.geojson`** — the same geometry in WGS84 (EPSG:4326) for interoperability.

The CSV joins to the geometry layer on **`hex_id`** (the relational structure requested).
Reported zero stays `0`; missing data stays **empty** (never `0`).

*Requires:* `geopandas pandas numpy pyogrio` (`pip install ...`).

In [1]:
# ============================ CONFIG — edit these ============================
GPKG      = "hexgrid_harvest_8km.gpkg"   # path to the aggregated GeoPackage
OUT_CSV   = "harvest_density_long.csv"
OUT_GPKG  = "hexgrid_geometry.gpkg"
OUT_JSON  = "hexgrid_geometry.geojson"

# Restrict to the species/metrics in the paper (set SPECIES = None to keep ALL species).
SPECIES   = ["RedDeer","RoeDeer","WildBoar","FallowDeer","SikaDeerJapaneese","Mouflon"]
METRICS   = ["Bag","Spring"]        # Bag=harvest, Spring=census; add "Plan" to publish the plan too
DROP_ALLNULL = True                 # drop variables never reported anywhere
GZIP      = True                    # write .csv.gz (Zenodo-friendly, ~20x smaller); False = plain .csv
SPLIT_ANNUAL_PERIOD = False         # True = separate annual and multi-year CSVs
ROUND     = 6                       # density decimals; None = full precision
# ===========================================================================

In [2]:
import re, numpy as np, pandas as pd, geopandas as gpd, pyogrio, warnings
warnings.filterwarnings("ignore")

METRIC_LABEL = {"Plan":"plan", "Bag":"harvest", "Spring":"spring_count"}
PREFIXES = tuple(f"{m}_" for m in METRICS)

layers = [l[0] for l in pyogrio.list_layers(GPKG)]
annual = sorted(l for l in layers if re.fullmatch(r"harvest_hex_\d{4}", l))
period = [l for l in layers if re.fullmatch(r"harvest_hex_\d{4}_\d{4}", l)]
print("annual layers:", len(annual), "| period layers:", len(period))
assert annual, "no annual harvest_hex_<year> layers found — check GPKG path/naming"

annual layers: 20 | period layers: 3


## 1 · Melt each layer to long form (`hex_id` × variable × period)

In [3]:
def _wanted(col):
    p = col.split("_"); species = "_".join(p[1:-1])
    return col.startswith(PREFIXES) and (SPECIES is None or species in SPECIES)

def _parse(col):
    p = col.split("_"); return METRIC_LABEL.get(p[0], p[0]), "_".join(p[1:-1]), p[-1]

def melt_layer(name, label):
    g = gpd.read_file(GPKG, layer=name)
    mcols = [c for c in g.columns if _wanted(c)]
    df = g[["hex_id"] + mcols].copy()
    for c in mcols:
        df[c] = pd.to_numeric(df[c], errors="coerce")      # keep NULL as NaN (not 0)
    long = df.melt(id_vars="hex_id", var_name="_c", value_name="density_ind_km2")
    t = long["_c"].map(_parse)
    long["metric"]        = t.map(lambda x: x[0])
    long["species"]       = t.map(lambda x: x[1])
    long["sex_age_class"] = t.map(lambda x: x[2])
    long["period"]        = label
    if ROUND is not None:
        long["density_ind_km2"] = long["density_ind_km2"].round(ROUND)
    return long[["hex_id","period","metric","species","sex_age_class","density_ind_km2"]]

annual_long = pd.concat([melt_layer(l, l.replace("harvest_hex_","")) for l in annual], ignore_index=True)
period_long = pd.concat([melt_layer(l, l.replace("harvest_hex_","")) for l in period], ignore_index=True) if period else annual_long.iloc[:0]
tidy = pd.concat([annual_long, period_long], ignore_index=True)
print(f"{len(tidy):,} rows before pruning")

2,103,350 rows before pruning


## 2 · Drop never-reported variables, then write the CSV(s)

In [4]:
def prune(df):
    if not DROP_ALLNULL: return df
    keep = df.groupby(["metric","species","sex_age_class"])["density_ind_km2"].transform(lambda s: s.notna().any())
    return df[keep]

def write_csv(df, path):
    p = path + (".gz" if GZIP else "")
    df.to_csv(p, index=False, na_rep="", compression=("gzip" if GZIP else None))  # empty = no data, 0 = reported zero
    print(f"wrote {p}: {len(df):,} rows, "
          f"{df[['metric','species','sex_age_class']].drop_duplicates().shape[0]} variables, "
          f"{df['period'].nunique()} periods")

if SPLIT_ANNUAL_PERIOD:
    write_csv(prune(annual_long), OUT_CSV.replace(".csv","_annual.csv"))
    if len(period_long): write_csv(prune(period_long), OUT_CSV.replace(".csv","_periods.csv"))
else:
    write_csv(prune(tidy), OUT_CSV)

wrote harvest_density_long.csv.gz: 2,103,350 rows, 50 variables, 23 periods


## 3 · Geometry-only hexagon layer (native 5514 + WGS84 GeoJSON)

In [5]:
geo = gpd.read_file(GPKG, layer=annual[0])                 # geometry identical across years
geo = geo[["hex_id"] + [c for c in ("area_km2","area_cz_km2") if c in geo.columns] + ["geometry"]]
geo.to_file(OUT_GPKG, layer="hexgrid", driver="GPKG")     # native CRS (EPSG:5514)
geo.to_crs(4326).to_file(OUT_JSON, driver="GeoJSON")      # WGS84 for interoperability
print(f"wrote {OUT_GPKG} (EPSG:{geo.crs.to_epsg()}) and {OUT_JSON} (EPSG:4326): {len(geo)} hexagons")

wrote hexgrid_geometry.gpkg (EPSG:5514) and hexgrid_geometry.geojson (EPSG:4326): 1829 hexagons


## 4 · Quick integrity check

In [6]:
chk = pd.read_csv(OUT_CSV + (".gz" if GZIP else ""), low_memory=False)
print("species :", sorted(chk.species.unique()))
print("metrics :", sorted(chk.metric.unique()), "| periods:", chk.period.nunique())
print("no-data (empty) cells:", int(chk.density_ind_km2.isna().sum()),
      "| reported zeros:", int((chk.density_ind_km2 == 0).sum()),
      "| positive:", int((chk.density_ind_km2 > 0).sum()))
print("\nTo map: load hexgrid_geometry.gpkg and merge your filtered CSV rows on 'hex_id'.")

species : ['FallowDeer', 'Mouflon', 'RedDeer', 'RoeDeer', 'SikaDeerJapaneese', 'WildBoar']
metrics : ['harvest', 'spring_count'] | periods: 23
no-data (empty) cells: 111557 | reported zeros: 720675 | positive: 1271118

To map: load hexgrid_geometry.gpkg and merge your filtered CSV rows on 'hex_id'.


### Notes
- **Join model:** CSV carries no geometry; join to `hexgrid_geometry` on `hex_id` (geometry stored once, not per year).
- **Null vs zero:** empty cell = no data; `0` = reported zero — preserved throughout.
- **Scope:** `SPECIES=None` exports all 38 species (much larger). Add `"Plan"` to `METRICS` to include the plan variable.
- **`period` column:** year (`2022`) for annual rows, range (`2013_2022`) for multi-year rows — it is a label, so it reads as text.

In [7]:
"""
Drop all Plan_* (hunting-plan) columns from every layer of the aggregated GeoPackage,
keeping harvest (Bag_*), census (Spring_*), and the diagnostic/geometry fields.
Writes a new GeoPackage; nulls and CRS are preserved (CRS stamped as ESRI WKT1 for ArcGIS).
Requires: geopandas, pyogrio
"""
import sqlite3, geopandas as gpd, pyogrio, warnings
from pyproj import CRS
warnings.filterwarnings("ignore")

IN_GPKG   = "hexgrid_harvest_8km.gpkg"
OUT_GPKG  = "hexgrid_harvest_8km_no_plan.gpkg"
DROP_PREFIX = "Plan_"
TARGET_CRS = 5514

import os
if os.path.exists(OUT_GPKG): os.remove(OUT_GPKG)

layers = [l[0] for l in pyogrio.list_layers(IN_GPKG)]
for name in layers:
    g = gpd.read_file(IN_GPKG, layer=name)
    drop = [c for c in g.columns if c.startswith(DROP_PREFIX)]
    g = g.drop(columns=drop)
    g.to_file(OUT_GPKG, layer=name, driver="GPKG")
    print(f"{name}: dropped {len(drop)} Plan_ columns -> {len(g.columns)-1} fields kept")

# stamp CRS as ESRI WKT1 so ArcGIS renders Krovák correctly
esri = CRS.from_epsg(TARGET_CRS).to_wkt(version="WKT1_ESRI")
con = sqlite3.connect(OUT_GPKG)
con.execute("UPDATE gpkg_spatial_ref_sys SET definition=? WHERE srs_id=? OR organization_coordsys_id=?",
            (esri, TARGET_CRS, TARGET_CRS))
con.commit(); con.close()
print(f"\nwrote {OUT_GPKG} ({len(layers)} layers, CRS EPSG:{TARGET_CRS} as ESRI WKT1)")

harvest_hex_2003: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2004: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2005: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2006: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2007: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2008: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2009: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2010: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2011: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2012: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2013: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2014: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2015: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2016: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2017: dropped 52 Plan_ columns -> 113 fields kept
harvest_hex_2018: dropped 52 Plan_ columns -> 113 fields kept
harvest_